<center><h1>Soundalgekar_Abhishek_Project_EfficientNetb0</h1></center>

Name: Abhishek Soundalgekar
<br>
Github Username: ABHISHEK SOUNDALGEKAR
<br>
USC ID: 2089011000

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.metrics import classification_report
from pathlib import Path
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from typing import Tuple
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    BatchNormalization,
    Dropout,
    Activation
)

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers     import GlobalAveragePooling2D, Dense, BatchNormalization, Dropout, Activation
from tensorflow.keras.models     import Model
from tensorflow.keras.optimizers import Adam as AdamOpt
from tensorflow.keras.callbacks  import EarlyStopping
from tensorflow.keras.regularizers import l2

2025-07-27 02:02:46.247718: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
DATA_ROOT: Path        = Path("../data/WasteSplit")
IMAGE_SHAPE: Tuple[int, int] = (224, 224)
BATCH_SIZE: int        = 5
EPOCH_COUNT: int       = 100

#EfficientNet‑Style Preprocessor
effnet_preprocessor = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.efficientnet.preprocess_input,
    rotation_range=25,            # random rotations up to ±25°
    width_shift_range=0.2,        # horizontal shifts
    height_shift_range=0.2,       # vertical shifts
    zoom_range=0.15,              # random zooms
    shear_range=0.1,              # slight shearing
    brightness_range=(0.85, 1.15),# brightness jitter
    horizontal_flip=True          # random flips
)

# Loader Factory
def make_data_loader(split: str, shuffle: bool = False):
    """
    Creates an iterator for one of ['train','val','test'].
    """
    folder = DATA_ROOT / split
    return effnet_preprocessor.flow_from_directory(
        directory=str(folder),
        target_size=IMAGE_SHAPE,
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        shuffle=shuffle
    )
train_loader = make_data_loader("train", shuffle=True)
val_loader   = make_data_loader("val",   shuffle=False)
test_loader  = make_data_loader("test",  shuffle=False)


Found 3323 images belonging to 9 classes.
Found 710 images belonging to 9 classes.
Found 719 images belonging to 9 classes.


In [3]:
pip install -U efficientnet


Note: you may need to restart the kernel to use updated packages.


In [4]:
from pathlib import Path
import efficientnet.tfkeras as efn
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.regularizers import l2

# --- Configuration ---
ROOT = Path("../data/WasteSplit")
H, W = 224, 224
BATCH, EPOCHS = 5, 100
LR, L2_REG, PATIENCE = 1e-4, 1e-2, 40

# --- Explicit Generators (no validation_split) ---
def make_gen(subdir, augment=False):
    folder = ROOT / subdir
    if augment:
        gen = ImageDataGenerator(
            preprocessing_function=tf.keras.applications.resnet.preprocess_input,
            rotation_range=20, width_shift_range=0.2, height_shift_range=0.2,
            zoom_range=0.2, brightness_range=(0.8,1.2), horizontal_flip=True
        )
    else:
        gen = ImageDataGenerator(
            preprocessing_function=tf.keras.applications.resnet.preprocess_input
        )
    return gen.flow_from_directory(
        directory=str(folder),
        target_size=(H, W),
        batch_size=BATCH,
        class_mode="categorical",
        shuffle=(subdir=="train")
    )

train_loader = make_gen("train", augment=True)
val_loader   = make_gen("val",   augment=True)
test_loader  = make_gen("test",  augment=False)

# --- Build ResNet101 + head ---
backbone = tf.keras.applications.ResNet101(
    weights="imagenet", include_top=False, input_shape=(H,W,3), name="resnet101_base"
)
backbone.trainable = False

def block(x, units, tag):
    x = layers.Dense(units, kernel_regularizer=l2(L2_REG), name=f"{tag}_dense")(x)
    x = layers.BatchNormalization(name=f"{tag}_bn")(x)
    return layers.Activation("relu", name=f"{tag}_act")(x)

x = layers.GlobalAveragePooling2D(name="gap")(backbone.output)
x = block(x, 256, "blk1")
x = block(x, 128, "blk2")
x = layers.Dropout(0.2, name="drop")(x)
x = block(x,  64, "blk3")

output = layers.Dense(
    train_loader.num_classes,
    activation="softmax",
    name="predictions"
)(x)

model = models.Model(backbone.input, output, name="ResNet101_Waste")
model.compile(
    optimizer=optimizers.Adam(LR),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# --- Train with EarlyStopping ---
es = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=PATIENCE,
    restore_best_weights=True,
    verbose=1
)
history = model.fit(
    train_loader,
    validation_data=val_loader,
    epochs=EPOCHS,
    callbacks=[es]
)


Found 3323 images belonging to 9 classes.
Found 710 images belonging to 9 classes.
Found 719 images belonging to 9 classes.


/opt/anaconda3/lib/python3.12/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 565s 830ms/step - accuracy: 0.2459 - loss: 9.0481 - val_accuracy: 0.6155 - val_loss: 7.7100
Epoch 2/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 560s 843ms/step - accuracy: 0.5244 - loss: 7.7850 - val_accuracy: 0.6915 - val_loss: 6.9011
Epoch 3/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 555s 835ms/step - accuracy: 0.5952 - loss: 7.0173 - val_accuracy: 0.7408 - val_loss: 6.2107
Epoch 4/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 552s 830ms/step - accuracy: 0.6164 - loss: 6.4210 - val_accuracy: 0.7761 - val_loss: 5.6057
Epoch 5/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 569s 855ms/step - accuracy: 0.6603 - loss: 5.7821 - val_accuracy: 0.7831 - val_loss: 5.1132
Epoch 6/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 581s 874ms/step - accuracy: 0.6758 - loss: 5.2895 - val_accuracy: 0.7915 - val_loss: 4.6292
Epoch 7/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 584s 879ms/step - accuracy: 0.6956 - loss: 4.8243 - val_accuracy: 0.7958 - val_loss: 4.2302
Epoch 8/100
665/665 ━━━━━━━━━━━━━━━━━━━━ 539s 811ms/step - accuracy: 

KeyboardInterrupt: 